# This file walks you through importing the data and saving it as a .npz to then work with in other sessions

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import src.util.dichasus_cf0x as dichasus_cf0x

4
<_ShardDataset element_spec=(TensorSpec(shape=(4, 2, 4, 64), dtype=tf.complex64, name=None), TensorSpec(shape=(3,), dtype=tf.float64, name=None), TensorSpec(shape=(), dtype=tf.float32, name=None))>


In [3]:

def load_data(tf_dataset, labeled = False):
    """
    Loads dataset from TensorFlow dataset to dictionary of NumPy arrays (might be more convenient)
    """
    data_dict = {
        "csi" : [],
        "time" : []
    }

    if labeled:
        data_dict["pos"] = []

    for datapoint in tf_dataset.batch(256):
        data_dict["csi"].append(datapoint[0].numpy())
        if labeled:
            data_dict["pos"].append(datapoint[1].numpy() if labeled else datapoint[0].numpy())
        data_dict["time"].append(datapoint[2].numpy() if labeled else datapoint[1].numpy())

    data_dict["csi"] = np.concatenate(data_dict["csi"])
    if labeled:
        data_dict["pos"] = np.concatenate(data_dict["pos"])
    data_dict["time"] = np.concatenate(data_dict["time"])

    return data_dict

In [4]:
training_data_labeled_np = load_data(dichasus_cf0x.training_set_labeled, labeled = True)
training_data_unlabeled_np = load_data(dichasus_cf0x.training_set_unlabeled)
test_data_np = load_data(dichasus_cf0x.test_set, labeled = True)

2025-05-20 15:46:05.948834: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:370] TFRecordDataset `buffer_size` is unspecified, default to 262144
2025-05-20 15:46:08.500666: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-05-20 15:46:17.052850: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [14]:
import importlib
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import src.util.dataset
importlib.reload(src.util.dataset)
from src.util.dataset import DichasusDataset, save_numpy_dataset

save_numpy_dataset(training_data_labeled_np, "training_data_labeled_np.npz")
save_numpy_dataset(training_data_unlabeled_np, "training_data_unlabeled_np.npz")
save_numpy_dataset(test_data_np, "test_data_np.npz")


'/Users/spencer/ISIT2025Competition/src/util/../../data/test_data_np.npz'

In [15]:
# Test to see if the data is loaded correctly
labelled_set = DichasusDataset(training_data_labeled_np)
unlabelled_set = DichasusDataset(training_data_unlabeled_np)
test_set = DichasusDataset(test_data_np)

print(labelled_set)
print(unlabelled_set)
print(test_set)
